In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Load dataset
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'
columns = ['checking', 'duration', 'credit_history', 'purpose', 'credit_amount', 'savings', 'employment', 
           'installment_rate', 'personal_status', 'other_debtors', 'residence', 'property', 'age', 
           'other_installment', 'housing', 'existing_credits', 'job', 'people_liable', 'telephone', 
           'foreign_worker', 'class']
df = pd.read_csv(url, sep=' ', names=columns)

# Data Exploration
print('Class Distribution:', df['class'].value_counts(normalize=True))
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='employment', hue='class')
plt.xticks(rotation=45)
plt.title('Credit Risk by Employment Duration')
plt.savefig('credit_plot.png', bbox_inches='tight')
plt.close()

# Data Cleaning and Feature Engineering
df['class'] = df['class'].map({1: 1, 2: 0})  # 1=Good, 2=Bad -> 1=Good, 0=Bad
df = pd.get_dummies(df, columns=['checking', 'credit_history', 'purpose', 'savings', 'employment', 
                                 'personal_status', 'other_debtors', 'property', 'other_installment', 
                                 'housing', 'job', 'telephone', 'foreign_worker'], drop_first=True)
# Create CreditToIncome feature (proxy income from savings and employment)
df['CreditToIncome'] = df['credit_amount'] / (df['savings_A62'] + df['savings_A63'] + df['savings_A64'] + df['savings_A65'] + 1)
df['credit_amount'] = df['credit_amount'].clip(upper=df['credit_amount'].quantile(0.95))

# Prepare data for modeling
X = df.drop('class', axis=1)
y = df['class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train and evaluate models
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Support Vector Machine': SVC(probability=True, random_state=42)
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    })

# Display results
results_df = pd.DataFrame(results)
print(results_df)

# Feature Importance for Random Forest
rf = models['Random Forest']
feature_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)[:10]
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importance.values, y=feature_importance.index)
plt.title('Top 10 Feature Importance (Random Forest)')
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.close()
